# Projections, Least Squares & Orthogonalization

投影、最小二乘与正交化。从投影矩阵到 Gram-Schmidt，从最小二乘几何解释到多种实现对比，全程配代码与可视化。

## 0. 环境配置与导入

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
print("PyTorch version:", torch.__version__)
torch.manual_seed(42)

## 1. 投影到一条直线（1D 子空间）

给定向量 $b$ 和一条过原点的直线（方向向量为 $a$），$b$ 在这条直线上的**正交投影** $p$ 是直线上离 $b$ 最近的点。

**公式**：

$$
p = \frac{a^T b}{a^T a} a = \hat{x} a
$$

其中 $\hat{x} = \frac{a^T b}{a^T a}$ 是投影系数。

**误差向量** $e = b - p$ 与直线方向 $a$ 正交：$a^T e = 0$。

**几何意义**：投影是将向量分解为"在子空间内的分量 $p$"和"与子空间正交的分量 $e$"，即 $b = p + e$，且 $p \perp e$。

In [ ]:
# 2D 中投影到一条直线
a = torch.tensor([2.0, 1.0])   # 直线方向
b = torch.tensor([1.0, 3.0])   # 待投影向量

# 投影系数
x_hat = torch.dot(a, b) / torch.dot(a, a)
# 投影向量
p = x_hat * a
# 误差向量
e = b - p

print("方向向量 a =", a.tolist())
print("待投影向量 b =", b.tolist())
print(f"\n投影系数 x_hat = (a·b)/(a·a) = {x_hat.item():.4f}")
print(f"投影向量 p = x_hat * a = {p.tolist()}")
print(f"误差向量 e = b - p = {e.tolist()}")
print(f"\n验证正交: a·e = {torch.dot(a, e).item():.6f} (≈0)")
print(f"验证分解: p + e = {(p+e).tolist()} == b? {torch.allclose(p+e, b)}")

# 可视化
fig, ax = plt.subplots(figsize=(6, 6))
# 直线
t = torch.linspace(-1, 3, 100)
ax.plot(t*a[0], t*a[1], 'b--', alpha=0.5, label='直线 span(a)')
ax.arrow(0, 0, a[0], a[1], head_width=0.15, color='blue', linewidth=2, label='a')
ax.arrow(0, 0, b[0], b[1], head_width=0.15, color='black', linewidth=2, label='b')
ax.arrow(0, 0, p[0], p[1], head_width=0.15, color='red', linewidth=2, label='p (投影)')
ax.plot([b[0], p[0]], [b[1], p[1]], 'g--', linewidth=1.5, label='e (误差, ⊥直线)')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_title("向量 b 投影到直线 span(a)")
ax.set_xlim(-0.5, 4); ax.set_ylim(-0.5, 4)
plt.tight_layout()
plt.show()

## 2. 投影到高维子空间与投影矩阵

### 2.1 投影到子空间

设 $A$ 的列张成子空间 $C(A) \subset \mathbb{R}^m$，向量 $b \in \mathbb{R}^m$。$b$ 在 $C(A)$ 上的正交投影 $p$ 满足：

$$
p = A\hat{x}, \quad \text{其中 } A^T(b - A\hat{x}) = 0
$$

即误差 $e = b - p$ 与 $C(A)$ 的所有列正交。

### 2.2 投影矩阵

从 $\hat{x} = (A^T A)^{-1} A^T b$（假设 A 列满秩）得：

$$
p = A(A^T A)^{-1} A^T b = P b
$$

其中 **投影矩阵**（projection matrix）：

$$
P = A(A^T A)^{-1} A^T
$$

投影矩阵将任意向量投影到 $C(A)$ 上。

In [ ]:
# 投影到 2D 平面（3D 中的子空间）
# A 的两列张成一个平面
A = torch.tensor([[1.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 1.0]])  # 3x2，列空间是 3D 中的一个平面
b = torch.tensor([1.0, 2.0, 3.0])  # 3D 向量

# 投影矩阵 P = A(A^T A)^{-1} A^T
ATA = A.T @ A
P = A @ torch.linalg.inv(ATA) @ A.T

print("A =\n", A)
print("\n投影矩阵 P = A(A^T A)^{-1} A^T =\n", P)

# 投影
p = P @ b
e = b - p

print("\nb =", b.tolist())
print("投影 p = P @ b =", p.tolist())
print("误差 e = b - p =", e.tolist())

# 验证：e 与 A 的每一列正交
print("\n验证正交:")
for j in range(A.shape[1]):
    print(f"  A[:,{j}] · e = {torch.dot(A[:,j], e).item():.6f} (≈0)")
print("  p + e =", (p+e).tolist(), "== b?", torch.allclose(p+e, b))

In [ ]:
# 投影矩阵的性质验证
A = torch.randn(5, 3)  # 5x3，列满秩
P = A @ torch.linalg.inv(A.T @ A) @ A.T

print("投影矩阵 P 的性质:")
print(f"  1. 对称性: P == P.T? {torch.allclose(P, P.T, atol=1e-5)}")
print(f"  2. 幂等性: P^2 == P? {torch.allclose(P @ P, P, atol=1e-5)}")
print(f"  3. 特征值只有 0 和 1: {torch.linalg.eigvalsh(P).round(decimals=4).tolist()}")
print(f"  4. rank(P) = rank(A) = {torch.linalg.matrix_rank(P).item()}")

# 验证：C(A) 中的向量投影后不变
x = torch.randn(3)
b_in_col = A @ x  # b 在列空间中
p_in_col = P @ b_in_col
print(f"\n5. 列空间中的向量投影后不变: P@(A@x) == A@x? {torch.allclose(p_in_col, b_in_col, atol=1e-5)}")

# 验证：左零空间中的向量投影后为 0
U, S, Vh = torch.linalg.svd(A)
left_null = U[:, 3:]  # 左零空间的基
b_in_null = left_null @ torch.randn(2)
p_in_null = P @ b_in_null
print(f"6. 左零空间中的向量投影后为0: ||P@b|| = {torch.norm(p_in_null).item():.6f} (≈0)")

## 3. 正交补与正交分解

### 3.1 正交补

子空间 $S$ 的**正交补**（orthogonal complement）$S^\perp$ 是与 $S$ 中所有向量都正交的向量集合：

$$
S^\perp = \{x \mid x^T s = 0, \forall s \in S\}
$$

**性质**：
- $S \cap S^\perp = \{0\}$
- $\dim(S) + \dim(S^\perp) = n$（全空间维度）
- $(S^\perp)^\perp = S$

对于矩阵 A：
- $C(A)^\perp = N(A^T)$（列空间的正交补 = 左零空间）
- $C(A^T)^\perp = N(A)$（行空间的正交补 = 零空间）

### 3.2 正交分解定理

> 任意向量 $b \in \mathbb{R}^m$ 可以唯一分解为 $b = p + e$，其中 $p \in C(A)$，$e \in N(A^T)$，且 $p \perp e$。

这就是投影的本质：将向量分解为"在子空间内"和"与子空间正交"两部分。

In [ ]:
# 正交分解：b = p + e, p ∈ C(A), e ∈ N(A^T), p⊥e
A = torch.randn(4, 2)  # 4x2，列空间维度 2，左零空间维度 2
b = torch.randn(4)

# 投影到列空间
P = A @ torch.linalg.inv(A.T @ A) @ A.T
p = P @ b
e = b - p

# 左零空间的基
U, S, Vh = torch.linalg.svd(A)
left_null_basis = U[:, 2:]  # [4, 2]

print("A: 4x2, rank =", torch.linalg.matrix_rank(A).item())
print("列空间维度 = 2, 左零空间维度 = 2")
print(f"\nb = {b.tolist()}")
print(f"p (列空间分量) = {p.tolist()}")
print(f"e (左零空间分量) = {e.tolist()}")

# 验证 p 在列空间中：p = A @ x 有解
x_p = torch.linalg.lstsq(A, p).solution
print(f"\np ∈ C(A)? A@x ≈ p? {torch.allclose(A @ x_p, p, atol=1e-5)}")

# 验证 e 在左零空间中：A^T @ e = 0
print(f"e ∈ N(A^T)? A^T @ e = {(A.T @ e).tolist()} (≈0)? {torch.allclose(A.T @ e, torch.zeros(2), atol=1e-5)}")

# 验证正交
print(f"\np · e = {torch.dot(p, e).item():.8f} (≈0, 正交)")
print(f"p + e = {(p+e).tolist()} == b? {torch.allclose(p+e, b)}")

# e 可以表示为左零空间基的线性组合
coeffs = left_null_basis.T @ e
e_reconstructed = left_null_basis @ coeffs
print(f"\ne = 左零空间基的线性组合? {torch.allclose(e_reconstructed, e, atol=1e-5)}")
print(f"系数: {coeffs.tolist()}")

## 4. Gram-Schmidt 正交化

Gram-Schmidt 正交化将一组线性无关的向量转化为一组标准正交向量，同时保持相同的张成空间。

**经典 Gram-Schmidt 算法**：
给定 $a_1, a_2, \ldots, a_n$：
1. $q_1 = a_1 / \|a_1\|$
2. 对 $j = 2, \ldots, n$：
   - $v_j = a_j - \sum_{i=1}^{j-1} (q_i^T a_j) q_i$（减去在已有正交向量上的投影）
   - $q_j = v_j / \|v_j\|$

**与 QR 分解的关系**：$A = QR$，其中 Q 的列是标准正交向量，R 是上三角矩阵，记录了投影系数。

$$
R_{ij} = q_i^T a_j \quad (i \leq j)
$$

In [ ]:
def gram_schmidt(A):
    """经典 Gram-Schmidt 正交化，返回 Q 和 R"""
    m, n = A.shape
    Q = torch.zeros(m, n)
    R = torch.zeros(n, n)
    
    for j in range(n):
        v = A[:, j].clone()
        # 减去在之前所有正交向量上的投影
        for i in range(j):
            R[i, j] = torch.dot(Q[:, i], A[:, j])
            v = v - R[i, j] * Q[:, i]
        # 归一化
        R[j, j] = torch.linalg.norm(v)
        if R[j, j] > 1e-10:
            Q[:, j] = v / R[j, j]
        else:
            Q[:, j] = 0  # 线性相关
    
    return Q, R

# 测试
A = torch.tensor([[1.0, 1.0, 1.0],
                  [1.0, 2.0, 3.0],
                  [1.0, 3.0, 6.0]])  # 希尔伯特矩阵型

Q_gs, R_gs = gram_schmidt(A)
Q_torch, R_torch = torch.linalg.qr(A)

print("A =\n", A)
print("\n手动 Gram-Schmidt:")
print("Q =\n", Q_gs)
print("R =\n", R_gs)
print("\nPyTorch QR:")
print("Q =\n", Q_torch)
print("R =\n", R_torch)

# 验证
print("\n验证:")
print(f"  Q 列正交: Q^T Q = I? {torch.allclose(Q_gs.T @ Q_gs, torch.eye(3), atol=1e-5)}")
print(f"  A = Q@R? {torch.allclose(Q_gs @ R_gs, A, atol=1e-5)}")
print(f"  R 上三角? {torch.allclose(R_gs, torch.triu(R_gs), atol=1e-5)}")
# 注意：QR 分解符号不唯一，Q 的列和 R 的行可以同时变号
print(f"  与 torch QR 结果一致（绝对值）? {torch.allclose(Q_gs.abs(), Q_torch.abs(), atol=1e-5)}")

In [ ]:
# Gram-Schmidt 的几何过程可视化（2D）
a1 = torch.tensor([2.0, 1.0])
a2 = torch.tensor([1.0, 3.0])

# Step 1: q1 = a1 / ||a1||
q1 = a1 / torch.linalg.norm(a1)
# Step 2: v2 = a2 - (q1·a2)q1, q2 = v2/||v2||
v2 = a2 - torch.dot(q1, a2) * q1
q2 = v2 / torch.linalg.norm(v2)

print("a1 =", a1.tolist(), ", a2 =", a2.tolist())
print("q1 =", q1.tolist(), "(单位向量)")
print("q1·a2 =", torch.dot(q1, a2).item())
print("v2 = a2 - (q1·a2)q1 =", v2.tolist())
print("q2 =", q2.tolist())
print("\n验证: q1·q2 =", torch.dot(q1, q2).item(), "(≈0, 正交)")
print("||q1|| =", torch.linalg.norm(q1).item(), ", ||q2|| =", torch.linalg.norm(q2).item())

# 可视化
fig, ax = plt.subplots(figsize=(7, 7))
ax.arrow(0, 0, a1[0], a1[1], head_width=0.15, color='blue', linewidth=2, label='a1')
ax.arrow(0, 0, a2[0], a2[1], head_width=0.15, color='green', linewidth=2, label='a2')
ax.arrow(0, 0, q1[0], q1[1], head_width=0.15, color='red', linewidth=2, label='q1 (a1归一化)')
ax.arrow(0, 0, q2[0], q2[1], head_width=0.15, color='purple', linewidth=2, label='q2 (正交化)')
# 投影虚线
proj_a2_on_q1 = torch.dot(q1, a2) * q1
ax.plot([a2[0], proj_a2_on_q1[0]], [a2[1], proj_a2_on_q1[1]], 'k--', alpha=0.5, label='a2在q1上的投影')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_title("Gram-Schmidt 正交化过程")
ax.set_xlim(-0.5, 3.5); ax.set_ylim(-0.5, 3.5)
plt.tight_layout()
plt.show()

## 5. 最小二乘法：几何解释与正规方程

### 5.1 问题描述

对于超定方程组 $Ax = b$（$m > n$，方程数多于未知数），通常无解。我们寻找使残差平方和最小的 $\hat{x}$：

$$
\hat{x} = \arg\min_x \|Ax - b\|_2^2
$$

### 5.2 几何解释

$Ax$ 始终在 $C(A)$ 中。要使 $\|Ax - b\|$ 最小，就是在 $C(A)$ 中找离 $b$ 最近的点——这正是 $b$ 在 $C(A)$ 上的正交投影 $p$！

因此：
- $p = A\hat{x}$ 是 $b$ 在 $C(A)$ 上的投影
- 残差 $e = b - p$ 与 $C(A)$ 正交，即 $A^T e = 0$

### 5.3 正规方程

由 $A^T(b - A\hat{x}) = 0$ 得**正规方程**（normal equations）：

$$
A^T A \hat{x} = A^T b
$$

当 A 列满秩时，$A^T A$ 可逆，解为：

$$
\hat{x} = (A^T A)^{-1} A^T b
$$

In [ ]:
# 最小二乘：几何解释
# 拟合直线 y = mx + c（2 个参数，多个数据点）
A = torch.tensor([[1.0, 1.0],   # [x, 1]，第一列是 x，第二列是常数项
                  [2.0, 1.0],
                  [3.0, 1.0],
                  [4.0, 1.0]])
b = torch.tensor([1.5, 2.8, 3.2, 4.5])  # 观测值（带噪声）

print("设计矩阵 A (4x2):")
print(A)
print("\n观测值 b:", b.tolist())

# 方法1：正规方程 x = (A^T A)^{-1} A^T b
ATA = A.T @ A
ATb = A.T @ b
x_normal = torch.linalg.inv(ATA) @ ATb
print("\n=== 正规方程 ===")
print("A^T A =\n", ATA)
print("A^T b =", ATb.tolist())
print("解 x = (A^T A)^{-1} A^T b =", x_normal.tolist())
print(f"即: 斜率 m = {x_normal[0].item():.4f}, 截距 c = {x_normal[1].item():.4f}")

# 投影 p = A @ x
p = A @ x_normal
e = b - p
print("\n投影 p = A@x =", p.tolist())
print("残差 e = b-p =", e.tolist())
print(f"残差平方和 = {torch.dot(e, e).item():.4f}")

# 验证：e 与 A 的列正交（A^T e = 0）
print("\n验证正交: A^T e =", (A.T @ e).tolist(), "(≈0)")

# 可视化
fig, ax = plt.subplots(figsize=(8, 6))
x_data = A[:, 0].numpy()
ax.scatter(x_data, b.numpy(), c='blue', s=80, zorder=5, label='数据点')
x_line = torch.linspace(0, 5, 100)
y_line = x_normal[0] * x_line + x_normal[1]
ax.plot(x_line.numpy(), y_line.numpy(), 'r-', linewidth=2, label=f'最小二乘拟合: y={x_normal[0]:.3f}x+{x_normal[1]:.3f}')
# 残差线
for i in range(len(x_data)):
    ax.plot([x_data[i], x_data[i]], [b[i], p[i]], 'g--', alpha=0.7, linewidth=1)
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_title("最小二乘线性拟合（绿色虚线=残差）")
plt.tight_layout()
plt.show()

## 6. 最小二乘的多种实现对比

求解最小二乘有多种方法，各有优劣：

| 方法 | 公式 | 优点 | 缺点 |
|------|------|------|------|
| 正规方程 | $(A^T A)^{-1} A^T b$ | 简单直观 | 数值不稳定（条件数平方），A 列秩亏时失败 |
| QR 分解 | $R^{-1} Q^T b$ | 数值稳定 | 需要 QR 分解 |
| SVD / 伪逆 | $V \Sigma^+ U^T b$ | 最稳定，支持秩亏 | 计算量最大 |
| `linalg.lstsq` | （内部自动选择） | 推荐使用 | - |

**数值稳定性**：当 A 的条件数很大时，正规方程的条件数是 A 的平方，误差被放大。QR 和 SVD 方法更稳定。

In [ ]:
# 四种最小二乘实现对比
torch.manual_seed(42)
m, n = 100, 5
A = torch.randn(m, n)
x_true = torch.randn(n)
b = A @ x_true + 0.1 * torch.randn(m)  # 加噪声

# 方法1：正规方程
x1 = torch.linalg.inv(A.T @ A) @ A.T @ b

# 方法2：QR 分解
Q, R = torch.linalg.qr(A)
x2 = torch.linalg.solve(R, Q.T @ b)  # Rx = Q^T b

# 方法3：SVD / 伪逆
x3 = torch.linalg.pinv(A) @ b

# 方法4：linalg.lstsq（推荐）
x4, _, _, _ = torch.linalg.lstsq(A, b)

print(f"真实 x: {x_true.tolist()}")
print(f"\n四种方法的解:")
print(f"  正规方程:  {x1.tolist()}")
print(f"  QR 分解:   {x2.tolist()}")
print(f"  SVD/伪逆:  {x3.tolist()}")
print(f"  lstsq:     {x4.tolist()}")

print(f"\n与真实值的误差（||x - x_true||）:")
print(f"  正规方程:  {torch.norm(x1 - x_true).item():.6f}")
print(f"  QR 分解:   {torch.norm(x2 - x_true).item():.6f}")
print(f"  SVD/伪逆:  {torch.norm(x3 - x_true).item():.6f}")
print(f"  lstsq:     {torch.norm(x4 - x_true).item():.6f}")

print(f"\n四种方法结果一致?")
print(f"  正规==QR:   {torch.allclose(x1, x2, atol=1e-4)}")
print(f"  QR==SVD:    {torch.allclose(x2, x3, atol=1e-4)}")
print(f"  SVD==lstsq: {torch.allclose(x3, x4, atol=1e-4)}")

In [ ]:
# 病态矩阵：正规方程 vs QR vs SVD 的数值稳定性对比
# 构造条件数很大的矩阵
torch.manual_seed(42)
m, n = 50, 5
U_ill, _, Vh_ill = torch.linalg.svd(torch.randn(m, n))
# 奇异值从 1 到 1e-8，条件数 1e8
singular_values = torch.tensor([1.0, 0.1, 0.01, 0.001, 1e-6])
A_ill = U_ill[:, :n] @ torch.diag(singular_values) @ Vh_ill[:n, :]

x_true = torch.randn(n)
b = A_ill @ x_true + 1e-4 * torch.randn(m)

cond_A = singular_values[0] / singular_values[-1]
cond_ATA = cond_A ** 2
print(f"A 的条件数 ≈ {cond_A.item():.0f}")
print(f"A^T A 的条件数 ≈ {cond_ATA.item():.0f}（正规方程误差被放大！）")

x_normal = torch.linalg.inv(A_ill.T @ A_ill) @ A_ill.T @ b
Q, R = torch.linalg.qr(A_ill)
x_qr = torch.linalg.solve(R, Q.T @ b)
x_svd = torch.linalg.pinv(A_ill) @ b
x_lstsq, _, _, _ = torch.linalg.lstsq(A_ill, b)

print(f"\n真实 x: {x_true.tolist()}")
print(f"\n误差 ||x - x_true||:")
print(f"  正规方程:  {torch.norm(x_normal - x_true).item():.6f} (最差!)")
print(f"  QR 分解:   {torch.norm(x_qr - x_true).item():.6f}")
print(f"  SVD/伪逆:  {torch.norm(x_svd - x_true).item():.6f}")
print(f"  lstsq:     {torch.norm(x_lstsq - x_true).item():.6f} (最好)")
print("\n→ 病态矩阵下，正规方程误差远大于 QR/SVD，实际中推荐用 lstsq")

## 7. 应用：线性回归拟合（含多项式回归）

最小二乘最广泛的应用是线性回归。对于非线性关系，可以通过**基函数展开**转化为线性回归：

- 线性回归：$y = w_0 + w_1 x$ → 设计矩阵 $[1, x]$
- 多项式回归：$y = w_0 + w_1 x + w_2 x^2 + \cdots + w_d x^d$ → 设计矩阵 $[1, x, x^2, \ldots, x^d]$

虽然模型对 x 是非线性的，但对参数 w 是线性的，因此仍可用最小二乘求解。

In [ ]:
# 多项式回归：用不同阶数拟合非线性数据
torch.manual_seed(42)
n_samples = 50
x = torch.linspace(-3, 3, n_samples)
# 真实关系：y = sin(x) + 0.1*x + 噪声
y_true = torch.sin(x) + 0.1 * x
y = y_true + 0.2 * torch.randn(n_samples)

def polynomial_features(x, degree):
    """构造多项式特征矩阵 [1, x, x^2, ..., x^degree]"""
    return torch.stack([x ** i for i in range(degree + 1)], dim=1)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()
degrees = [1, 2, 4, 8]

for idx, degree in enumerate(degrees):
    ax = axes[idx]
    # 构造设计矩阵
    X_poly = polynomial_features(x, degree)
    # 最小二乘求解
    w, _, _, _ = torch.linalg.lstsq(X_poly, y)
    # 预测
    x_dense = torch.linspace(-3, 3, 200)
    y_pred = polynomial_features(x_dense, degree) @ w
    
    ax.scatter(x.numpy(), y.numpy(), c='blue', alpha=0.5, s=30, label='带噪声数据')
    ax.plot(x_dense.numpy(), y_pred.numpy(), 'r-', linewidth=2, label=f'{degree}阶多项式拟合')
    ax.plot(x.numpy(), y_true.numpy(), 'g--', alpha=0.7, label='真实函数 sin(x)+0.1x')
    ax.set_title(f"多项式回归 degree={degree}")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.set_ylim(-2, 2)

plt.suptitle("多项式回归：阶数越高，拟合能力越强（但可能过拟合）", fontsize=14)
plt.tight_layout()
plt.show()

# 计算各阶数的训练误差和测试误差
print("\n各阶数的训练误差（MSE）:")
for degree in [1, 2, 3, 4, 6, 8, 12]:
    X_poly = polynomial_features(x, degree)
    w, _, _, _ = torch.linalg.lstsq(X_poly, y)
    y_pred = X_poly @ w
    mse = torch.mean((y - y_pred) ** 2).item()
    print(f"  degree={degree:>2}: MSE = {mse:.6f}")

In [ ]:
# 正则化最小二乘（岭回归 Ridge Regression）
# 当特征高度相关或阶数过高时，正规方程 A^T A 可能接近奇异
# 岭回归加入 L2 正则项：min ||Ax-b||^2 + λ||x||^2
# 解为：x = (A^T A + λI)^{-1} A^T b

torch.manual_seed(42)
degree = 12
X_poly = polynomial_features(x, degree)

print(f"高阶多项式回归 (degree={degree})")
print(f"A^T A 的条件数: {torch.linalg.cond(X_poly.T @ X_poly).item():.2e}")

# 不同正则化强度
lambdas = [0.0, 0.001, 0.01, 0.1, 1.0]
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(x.numpy(), y.numpy(), c='blue', alpha=0.5, s=30, label='数据')

x_dense = torch.linspace(-3, 3, 200)
X_dense = polynomial_features(x_dense, degree)
colors = ['red', 'orange', 'green', 'blue', 'purple']

for lam, color in zip(lambdas, colors):
    # 岭回归解
    ATA_reg = X_poly.T @ X_poly + lam * torch.eye(degree + 1)
    w_ridge = torch.linalg.solve(ATA_reg, X_poly.T @ y)
    y_pred = X_dense @ w_ridge
    
    mse = torch.mean((y - X_poly @ w_ridge) ** 2).item()
    w_norm = torch.norm(w_ridge).item()
    ax.plot(x_dense.numpy(), y_pred.numpy(), color=color, linewidth=2,
            label=f'λ={lam}, MSE={mse:.4f}, ||w||={w_norm:.2f}')

ax.set_title("岭回归：λ 越大，参数越平滑（抑制过拟合）")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.set_ylim(-2, 2)
plt.tight_layout()
plt.show()

print("\n→ λ=0 是普通最小二乘（过拟合，振荡大）")
print("→ λ 增大，参数 ||w|| 减小，曲线更平滑（偏差增大，方差减小）")

## 课后练习

### 基础题

1. 给定向量 $b = (3, 4)$ 和直线方向 $a = (1, 2)$，手动计算 $b$ 在直线上的正交投影 $p$ 和误差 $e$，验证 $a \perp e$。用代码验证。

2. 对矩阵 $A = \begin{pmatrix} 1 & 0 \\ 1 & 1 \\ 0 & 1 \end{pmatrix}$，计算投影矩阵 $P = A(A^T A)^{-1} A^T$，验证 P 的对称性、幂等性，以及特征值只有 0 和 1。

3. 验证正交分解定理：取一个 4×2 矩阵 A 和随机向量 b，分解 b = p + e，验证 p ∈ C(A)、e ∈ N(A^T)、p ⊥ e。

### Gram-Schmidt

4. 手动实现修正 Gram-Schmidt（Modified Gram-Schmidt，比经典版数值更稳定），对一个 5×4 随机矩阵计算 Q 和 R，与 `torch.linalg.qr` 对比结果。

5. 用 Gram-Schmidt 证明：如果 A 列满秩，则 QR 分解中 R 的对角线元素为正。

### 最小二乘

6. 生成 100 个带噪声的数据点（真实关系为 $y = 2x + 1 + \text{noise}$），分别用正规方程、QR 分解、SVD 伪逆、`lstsq` 四种方法拟合直线，对比四种方法得到的斜率和截距。

7. 构造一个条件数为 $10^6$ 的病态矩阵 A（50×5），对比正规方程和 `lstsq` 在求解最小二乘时的误差，解释为什么正规方程误差更大。

8. 多项式回归：生成 $y = x^3 - 2x^2 + 0.5x + 1 + \text{noise}$ 的数据（100 个点），分别用 1、2、3、5、8 阶多项式拟合，画出拟合曲线和训练 MSE 随阶数的变化曲线，解释过拟合现象。

### 综合题

9. 证明投影矩阵的性质：如果 P 是投影到子空间 S 的投影矩阵，则 $I - P$ 是投影到正交补 $S^\perp$ 的投影矩阵。用代码验证一个 3D 例子。

10. 岭回归推导：从目标函数 $\min_x \|Ax - b\|^2 + \lambda\|x\|^2$ 出发，推导其解为 $x = (A^T A + \lambda I)^{-1} A^T b$。用一个多项式回归例子验证岭回归能抑制过拟合。